Feature engineering decisions

The goal of this notebook is to create a few model-ready features from the cleaned used-car dataset.

Features added in this notebook:
- `car_age` from `model_year` using `car_age = current_year - model_year`
- `scaled_mileage` from mileage using `scaled_mileage = mileage / 1000`
- `fuel_type_encoded` by mapping selected categories to integers
  - `Gas` -> `0`
  - `Diesel` -> `1`
  - `Electric` -> `2`

Original columns are preserved, and new engineered columns are added alongside them.

In [4]:
# Run this once in fresh notebook environments where pandas/numpy are missing.
%pip install pandas numpy -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

CURRENT_YEAR = pd.Timestamp.now().year

# Try notebook-local and project-root data paths for portability.
INPUT_CANDIDATES = [
    Path("cleaned_used_cars.csv"),
    Path("../data/cleaned_used_cars.csv"),
    Path("data/cleaned_used_cars.csv"),
]

## 1. Load the cleaned dataset

We load the cleaned CSV and preview the data before feature engineering begins.

In [6]:
def resolve_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find cleaned_used_cars.csv in expected locations: "
        + ", ".join(str(p) for p in candidates)
    )

input_path = resolve_existing_path(INPUT_CANDIDATES)
df = pd.read_csv(input_path)

print(f"Using input file: {input_path.resolve()}")
print(f"Rows, columns: {df.shape}")
display(df.head())

Using input file: /workspaces/COMP-3250-Used-Car-Price-Estimation/data/cleaned_used_cars.csv
Rows, columns: (4009, 16)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,price_usd,mileage_mi,clean_title_flag,accident_reported
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capability,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300",10300,51000,True,True
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005",38005,34742,True,True
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598",54598,22372,NaN,False
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric Hybrid,7-Speed A/T,Black,Black,None reported,Yes,"$15,500",15500,88900,True,False
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999",34999,9835,NaN,False


## 2. Convert model year to car age

The `model_year` column is converted into a more model-friendly feature called `car_age`.

Formula used:
- `car_age = current_year - model_year`

Implementation notes:
- `model_year` is coerced to numeric so invalid text becomes missing values.
- Future model years are treated as invalid for this feature and set to missing (`NaN`).

In [7]:
feat = df.copy()

feat["model_year_numeric"] = pd.to_numeric(feat["model_year"], errors="coerce")
feat["car_age"] = CURRENT_YEAR - feat["model_year_numeric"]

# Guard against negative ages from future model years.
feat.loc[feat["car_age"] < 0, "car_age"] = np.nan

age_preview = feat[["model_year", "model_year_numeric", "car_age"]].head(10)
display(age_preview)

,model_year,model_year_numeric,car_age
0,2013,2013,13.0
1,2021,2021,5.0
2,2022,2022,4.0
3,2015,2015,11.0
4,2021,2021,5.0
5,2016,2016,10.0
6,2017,2017,9.0
7,2001,2001,25.0
8,2021,2021,5.0
9,2020,2020,6.0


## 3. Scale mileage

Mileage can be large in raw numeric form, so we create `scaled_mileage` to express mileage in thousands of miles:

- `scaled_mileage = mileage / 1000`

If the cleaned file already has `mileage_mi`, it is used directly.
If not, mileage is extracted from `milage` and converted to numeric first.

In [8]:
if "mileage_mi" in feat.columns:
    mileage_base = pd.to_numeric(feat["mileage_mi"], errors="coerce")
else:
    mileage_base = pd.to_numeric(
        feat["milage"].astype(str).str.replace(r"[^0-9]", "", regex=True),
        errors="coerce",
    )

feat["scaled_mileage"] = mileage_base / 1000

mileage_preview = pd.DataFrame(
    {
        "original_mileage": mileage_base.head(10),
        "scaled_mileage": feat["scaled_mileage"].head(10),
    }
)
display(mileage_preview)

,original_mileage,scaled_mileage
0,51000,51.000
1,34742,34.742
2,22372,22.372
3,88900,88.900
4,9835,9.835
5,136397,136.397
6,84000,84.000
7,242000,242.000
8,23436,23.436
9,34000,34.000


## 4. Encode categorical values

For `fuel_type`, we apply the requested integer mapping:

- `Gas` -> `0`
- `Diesel` -> `1`
- `Electric` -> `2`

To avoid mismatches from capitalization, spacing, and common label variants, text is normalized before mapping.
For example, `Gasoline` is treated as `Gas`, and `Battery Electric` is treated as `Electric`.

In [12]:
fuel_map = {"gas": 0, "diesel": 1, "electric": 2}

fuel_normalized = feat["fuel_type"].astype(str).str.strip().str.lower()
fuel_normalized = fuel_normalized.replace({"nan": np.nan, "none": np.nan})

# Normalize common variants before encoding.
fuel_normalized = fuel_normalized.replace(
    {
        "gasoline": "gas",
        "battery electric": "electric",
        "electric motor": "electric",
    }
)

feat["fuel_type_encoded"] = fuel_normalized.map(fuel_map)

encoding_preview = pd.DataFrame(
    {
        "fuel_type": feat["fuel_type"],
        "fuel_type_encoded": feat["fuel_type_encoded"],
    }
)
display(
    encoding_preview.drop_duplicates()
    .sort_values("fuel_type", na_position="last")
    .head(20)
)

,fuel_type,fuel_type_encoded
42,Diesel,1.0
0,E85 Flex Fuel,NaN
1,Gasoline,0.0
3,Hybrid,NaN
60,Plug-In Hybrid,NaN
9,NaN,NaN


## 5. Validate engineered features

We run a quick check on the new columns to verify data types, missing values, and basic ranges.

In [13]:
new_cols = ["car_age", "scaled_mileage", "fuel_type_encoded"]

feature_audit = pd.DataFrame(
    {
        "dtype": feat[new_cols].dtypes.astype(str),
        "missing_values": feat[new_cols].isna().sum(),
        "missing_percent": (feat[new_cols].isna().mean() * 100).round(2),
        "min": feat[new_cols].min(numeric_only=True),
        "max": feat[new_cols].max(numeric_only=True),
    }
)

display(feature_audit)
display(feat[new_cols].head(10))

,dtype,missing_values,missing_percent,min,max
car_age,float64,0,0.00,2.0,52.0
scaled_mileage,float64,0,0.00,0.1,405.0
fuel_type_encoded,float64,584,14.57,0.0,1.0


,car_age,scaled_mileage,fuel_type_encoded
0,13.0,51.000,NaN
1,5.0,34.742,0.0
2,4.0,22.372,0.0
3,11.0,88.900,NaN
4,5.0,9.835,0.0
5,10.0,136.397,0.0
6,9.0,84.000,0.0
7,25.0,242.000,0.0
8,5.0,23.436,0.0
9,6.0,34.000,NaN


## 6. Save engineered dataset

Finally, we export the dataset with the new engineered features for model training and evaluation.

In [14]:
output_path = input_path.with_name("feature_engineered_used_cars.csv")
feat.to_csv(output_path, index=False)
print(f"Saved engineered file to: {output_path.resolve()}")

Saved engineered file to: /workspaces/COMP-3250-Used-Car-Price-Estimation/data/feature_engineered_used_cars.csv
